In [4]:
import pandas as pd
import pathlib
from alphagenome.data import genome
from alphagenome.interpretation import enhancer_scoring
from alphagenome.models.v2 import dna_client, dna_output

ROOT = pathlib.Path('/storage/mathelierarea/processed/eirikhoy/action_var')

API_KEY = (ROOT / 'resources/API_KEY').read_text().strip()

dna_model = dna_client.create(API_KEY)

In [6]:
gtf = pd.read_feather(
    "https://storage.googleapis.com/alphagenome/reference/gencode/hg38/"
    "gencode.v46.annotation.gtf.gz.feather"
    )

GENE_ID = "ENSG00000119866"  # BCL11A

conditions = [
    "UBERON:0002371_CL:0000765",  # bone marrow erythroblast
    "UBERON:0002371_CL:0000788",  # bone marrow naive B cell
]

In [7]:
# 1. Input gradients of gene's expression, one output per condition
grads = dna_model.predict_interval_gradients_batch(
    gene_ids=GENE_ID,
    requested_output=dna_output.OutputType.SC_GENE_EXPRESSION,
    sequence_length=dna_client.SEQUENCE_LENGTH_1MB,
    conditions=conditions,
)



  0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
# 2. Matching chromatin accessibility over the same intervals
atac = dna_model.predict_intervals(
    intervals=[g.interval for g in grads],
    conditions=conditions,
    requested_outputs=[dna_output.OutputType.SC_ATAC],
)



  0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# 3a. De novo enhancer calls (thresholded), one row per enhancer-gene link
enhancers = enhancer_scoring.get_active_enhancers(
    grads, gtf=gtf, accessibility_outputs=atac
)
print(enhancers[['biosample_name', 'chromosome', 'start', 'end',
                'tss_distance', 'gradient_score', 'atac_score']])

              biosample_name chromosome     start       end  tss_distance  \
0   bone marrow erythroblast       chr2  60439619  60440119      113785.0   
1   bone marrow erythroblast       chr2  60473619  60474119       79785.0   
2   bone marrow erythroblast       chr2  60490619  60491119       62785.0   
3   bone marrow erythroblast       chr2  60495119  60495619       58285.0   
4   bone marrow erythroblast       chr2  60498119  60498619       55285.0   
5   bone marrow erythroblast       chr2  60504119  60504619       49285.0   
6   bone marrow erythroblast       chr2  60525619  60526119       27785.0   
7   bone marrow erythroblast       chr2  60550119  60550619        3285.0   
8   bone marrow erythroblast       chr2  60551869  60552369        1535.0   
9   bone marrow erythroblast       chr2  60553369  60553869          35.0   
10  bone marrow naive B cell       chr2  60317869  60318369      235535.0   
11  bone marrow naive B cell       chr2  60336869  60337369      216535.0   

In [11]:
# 3b. Or score your own candidate regions (no thresholds)
candidates = [genome.Interval("chr2", 60_497_676, 60_498_941, name="+55kb")]
print(enhancer_scoring.score_enhancers(grads[0], candidates))

  chromosome     start       end   name            biosample_name  \
0       chr2  60497676  60498941  +55kb  bone marrow erythroblast   

              gene_id  gradient_score  is_scored gene_name  
0  ENSG00000119866.22        0.032665       True    BCL11A  
